# NB-11: VaceWanAttentionBlock -- VACE Control Block

## Learning Objectives
- Understand how VaceWanAttentionBlock inherits DiTBlock, adding only `before_proj` and `after_proj` Linear layers
- Trace the stack/unstack accumulation pattern step-by-step, seeing how the hint stack grows from `[2,B,S,D]` to `[4,B,S,D]` over 3 blocks
- Distinguish the live hidden state `c` from accumulated skip connections `c_skip` in the growing stack

## Prerequisites
- **Prior notebooks:** NB-06 (DiTBlock forward pass -- self-attention, cross-attention, FFN with adaLN-Zero)
- **Assumed concepts:** DiTBlock as a black box (`forward(x, context, t_mod, freqs) -> x`), inheritance, `torch.stack` / `torch.unbind`

## Concept Map
- **VaceWanAttentionBlock** wraps DiTBlock to produce control hints via stack/unstack accumulation -> NB-12 shows how these hints integrate into the DiT pipeline

In [ ]:
# Source: diffsynth/models/wan_video_vace.py (setup)
# Source: diffsynth/models/wan_video_dit.py (setup)
import sys
import types
import importlib.util
import pathlib

# Find project root: walk up from Course/ to find the directory containing diffsynth/
# This handles both normal checkout and git worktree scenarios.
_here = pathlib.Path().resolve()  # where notebook is executed from
PROJECT_ROOT = None
for _candidate in [_here] + list(_here.parents)[:6]:
    if (_candidate / "diffsynth" / "models" / "wan_video_dit.py").exists():
        PROJECT_ROOT = _candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find diffsynth/models/wan_video_dit.py. "
        "Run this notebook from Course/ inside the project checkout."
    )
print(f"Project root: {PROJECT_ROOT}")

# Stub wan_video_camera_controller (not needed for VACE demos)
_cam_stub = types.ModuleType("diffsynth.models.wan_video_camera_controller")
_cam_stub.SimpleAdapter = type("SimpleAdapter", (), {"__init__": lambda s, *a, **kw: None})
sys.modules["diffsynth"] = types.ModuleType("diffsynth")
sys.modules["diffsynth.models"] = types.ModuleType("diffsynth.models")
sys.modules["diffsynth.models.wan_video_camera_controller"] = _cam_stub

# Load DiT first (VACE depends on it)
_dit_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_dit.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_dit", _dit_path)
dit_mod = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_dit"] = dit_mod
_spec.loader.exec_module(dit_mod)

# Load VACE
_vace_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_vace.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_vace", _vace_path)
vace_mod = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_vace"] = vace_mod
_spec.loader.exec_module(vace_mod)

from diffsynth.models.wan_video_vace import VaceWanAttentionBlock, VaceWanModel
from diffsynth.models.wan_video_dit import DiTBlock, precompute_freqs_cis_3d
import torch

print("Setup complete.")

## VACE Accumulation Pattern -- Stack Grows Block-by-Block

```
VACE Accumulation: unbind -> pop -> DiTBlock -> after_proj -> append -> stack
=====================================================================

Block 0 (block_id=0):   -- has before_proj, others do NOT
  INPUT:  c [B, S, D]  (control tokens from patch embedding)
          x [B, S, D]  (noisy latent from DiT)
  Step 1: c = before_proj(c) + x        -> c [B, S, D]   (project + add noisy latent)
  Step 2: all_c = []                     -> empty list
  Step 3: c = super().forward(c, ...)    -> c [B, S, D]   (full DiTBlock forward)
  Step 4: c_skip = after_proj(c)         -> c_skip [B, S, D]
  Step 5: all_c += [c_skip, c]           -> [c_skip_0, c_0]
  Step 6: c = stack(all_c)              -> c [2, B, S, D]  <<< stack starts

Block 1 (block_id=1):
  INPUT:  c [2, B, S, D]
  Step 1: all_c = unbind(c)              -> [c_skip_0, c_0]  (split stack)
  Step 2: c = all_c.pop(-1)             -> c = c_0 [B, S, D] (pop live state)
          all_c = [c_skip_0]             -> accumulated skips remain
  Step 3: c = super().forward(c, ...)    -> c [B, S, D]
  Step 4: c_skip = after_proj(c)         -> c_skip_1 [B, S, D]
  Step 5: all_c += [c_skip_1, c]         -> [c_skip_0, c_skip_1, c_1]
  Step 6: c = stack(all_c)              -> c [3, B, S, D]  <<< stack grows

Block 2 (block_id=2):
  INPUT:  c [3, B, S, D]
  Step 1: all_c = unbind(c)              -> [c_skip_0, c_skip_1, c_1]
  Step 2: c = all_c.pop(-1)             -> c = c_1 [B, S, D]
          all_c = [c_skip_0, c_skip_1]
  Step 3: c = super().forward(c, ...)    -> c [B, S, D]
  Step 4: c_skip = after_proj(c)         -> c_skip_2 [B, S, D]
  Step 5: all_c += [c_skip_2, c]         -> [c_skip_0, c_skip_1, c_skip_2, c_2]
  Step 6: c = stack(all_c)              -> c [4, B, S, D]  <<< stack grows

FINAL:
  hints = unbind(c)[:-1]                 -> [c_skip_0, c_skip_1, c_skip_2]
                                            3 hints, each [B, S, D]
                                            (drop final live c_2)

Key insight: The LAST element in the stack is always the live hidden state.
             Everything before it is accumulated skip-projected hints.
```

Source: `diffsynth/models/wan_video_vace.py`, lines 13-24

## Brief DiTBlock Recap

DiTBlock (see NB-06) wires 5 primitives: RMSNorm -> self-attention -> cross-attention -> FFN, with adaLN-Zero modulation and gated residuals.

```python
def forward(self, x, context, t_mod, freqs) -> x  # [B, S, dim]
```

VaceWanAttentionBlock calls `super().forward()` which runs the full DiTBlock pipeline. We treat this as a black box here -- the internals are covered in NB-06.

Source: `diffsynth/models/wan_video_dit.py`, line 215

## Sub-Module Inventory

VaceWanAttentionBlock inherits **all** DiTBlock sub-modules and adds only two Linear projections:

| Sub-module | Type | Purpose | Present in DiTBlock? |
|------------|------|---------|---------------------|
| `self_attn` | SelfAttention | Token-to-token attention with RoPE | Yes (inherited) |
| `cross_attn` | CrossAttention | Text conditioning | Yes (inherited) |
| `ffn` | FFN | Feed-forward network | Yes (inherited) |
| `norm1, norm2, norm3` | LayerNorm | Layer normalization | Yes (inherited) |
| `modulation` | nn.Parameter | adaLN-Zero 6-param modulation | Yes (inherited) |
| `gate` | GateModule | adaLN-Zero gating | Yes (inherited) |
| `before_proj` | nn.Linear(384, 384) | Project control + add noisy latent (block_id=0 ONLY) | **NEW** |
| `after_proj` | nn.Linear(384, 384) | Project skip connection (ALL blocks) | **NEW** |

Source: `diffsynth/models/wan_video_vace.py`, lines 5-11

In [ ]:
# Source: diffsynth/models/wan_video_vace.py, lines 5-11
dim, num_heads, ffn_dim = 384, 4, 1024

block0 = VaceWanAttentionBlock(has_image_input=False, dim=dim, num_heads=num_heads, ffn_dim=ffn_dim, block_id=0)
block1 = VaceWanAttentionBlock(has_image_input=False, dim=dim, num_heads=num_heads, ffn_dim=ffn_dim, block_id=1)

p0 = sum(p.numel() for p in block0.parameters())
p1 = sum(p.numel() for p in block1.parameters())
print(f"block_id=0 params: {p0:,}")
print(f"block_id=1 params: {p1:,}")
print(f"Difference (before_proj): {p0 - p1:,}")

# Verify before_proj only exists on block_id=0
assert hasattr(block0, 'before_proj'), "block_id=0 must have before_proj"
assert not hasattr(block1, 'before_proj'), "block_id=1 must NOT have before_proj"
assert hasattr(block0, 'after_proj'), "block_id=0 must have after_proj"
assert hasattr(block1, 'after_proj'), "block_id=1 must have after_proj"

# Verify before_proj is dim->dim Linear
assert block0.before_proj.in_features == dim
assert block0.before_proj.out_features == dim
print(f"\nbefore_proj: Linear({dim}, {dim}) = {dim*dim + dim:,} params")
print(f"after_proj:  Linear({dim}, {dim}) = {dim*dim + dim:,} params")

## Inheritance Visualization

```
DiTBlock (NB-06)
  |
  +-- VaceWanAttentionBlock
        Adds: before_proj (block_id=0 only), after_proj (all blocks)
        Wraps: forward() with stack/unstack accumulation
        Inherits: ALL other sub-modules unchanged
```

Source: `diffsynth/models/wan_video_vace.py`, line 5

In [ ]:
# Source: diffsynth/models/wan_video_dit.py (precompute_freqs_cis_3d)
# Build RoPE frequencies for standalone block testing
head_dim = dim // num_heads  # 96
f_freqs, h_freqs, w_freqs = precompute_freqs_cis_3d(head_dim)
F, H, W = 4, 4, 4  # spatial grid -> S = 64
freqs = torch.cat([
    f_freqs[:F].view(F, 1, 1, -1).expand(F, H, W, -1),
    h_freqs[:H].view(1, H, 1, -1).expand(F, H, W, -1),
    w_freqs[:W].view(1, 1, W, -1).expand(F, H, W, -1),
], dim=-1).reshape(F * H * W, 1, -1)
S = F * H * W  # 64
assert freqs.shape == torch.Size([S, 1, head_dim // 2])
print(f"RoPE freqs: {freqs.shape}  (S={S}, head_dim//2={head_dim//2})")

In [ ]:
# Source: diffsynth/models/wan_video_vace.py, line 13 (forward inputs)
B, S, D = 1, 64, 384
c_input = torch.randn(B, S, D)   # initial control tokens (after patch embedding)
x = torch.randn(B, S, D)          # noisy latent tokens from DiT
context = torch.randn(B, 10, D)   # text context (10 tokens)
t_mod = torch.randn(B, 6, D)      # time modulation (6 params from adaLN-Zero)

assert c_input.shape == torch.Size([B, S, D])
assert x.shape == torch.Size([B, S, D])
assert context.shape == torch.Size([B, 10, D])
assert t_mod.shape == torch.Size([B, 6, D])

print(f"c_input (control tokens): {c_input.shape}")
print(f"x (noisy latent):         {x.shape}")
print(f"context (text):           {context.shape}")
print(f"t_mod (time modulation):  {t_mod.shape}")
print(f"freqs (RoPE):             {freqs.shape}")

## Step-by-Step Forward Trace

We now trace VaceWanAttentionBlock.forward() for each of the three blocks, verifying the accumulation pattern from the ASCII diagram above.

In [ ]:
# Source: diffsynth/models/wan_video_vace.py, lines 13-24
# ---- Block 0 (block_id=0): has before_proj ----
block0.eval()
with torch.no_grad():
    # Stage 1: before_proj(c) + x  (block_id=0 only)
    c0 = block0.before_proj(c_input) + x
    print(f"Stage 1 -- before_proj(c) + x:  {c0.shape}")
    assert c0.shape == torch.Size([B, S, D])

    # Stage 2: all_c = []  (empty list for block_id=0)
    all_c_0 = []
    print(f"Stage 2 -- all_c = []            len={len(all_c_0)}")

    # Stage 3: super().forward(c, context, t_mod, freqs)
    c0 = DiTBlock.forward(block0, c0, context, t_mod, freqs)
    print(f"Stage 3 -- super().forward():    {c0.shape}")
    assert c0.shape == torch.Size([B, S, D])

    # Stage 4: after_proj(c) -> c_skip
    c_skip_0 = block0.after_proj(c0)
    print(f"Stage 4 -- after_proj(c):        {c_skip_0.shape}  (c_skip_0)")
    assert c_skip_0.shape == torch.Size([B, S, D])

    # Stage 5: all_c += [c_skip, c]
    all_c_0 += [c_skip_0, c0]
    print(f"Stage 5 -- all_c += [c_skip, c]: len={len(all_c_0)}")

    # Stage 6: stack(all_c) -> [2, B, S, D]
    c_stacked_0 = torch.stack(all_c_0)
    print(f"Stage 6 -- stack(all_c):         {c_stacked_0.shape}")
    assert c_stacked_0.shape == torch.Size([2, B, S, D]), f"Expected [2,{B},{S},{D}], got {c_stacked_0.shape}"

print(f"\nBlock 0 complete: stack {c_stacked_0.shape}  [c_skip_0, c_0]")

In [ ]:
# Source: diffsynth/models/wan_video_vace.py, line 13
# Verify: direct call produces same output as manual trace
with torch.no_grad():
    c_direct_0 = block0(c_input, x, context, t_mod, freqs)
assert c_direct_0.shape == torch.Size([2, B, S, D])
assert torch.allclose(c_direct_0, c_stacked_0, atol=1e-6), "Manual trace must match direct call"
print(f"Verification: direct call matches manual trace  shape={c_direct_0.shape}")

In [ ]:
# Source: diffsynth/models/wan_video_vace.py, lines 13-24
# ---- Block 1 (block_id=1): unbind + pop branch ----
block1.eval()
with torch.no_grad():
    # Stage 1: unbind the stack from Block 0
    all_c_1 = list(torch.unbind(c_stacked_0))
    print(f"Stage 1 -- unbind(c):     {len(all_c_1)} tensors, each {all_c_1[0].shape}")
    assert len(all_c_1) == 2  # [c_skip_0, c_0]

    # Stage 2: pop last element (live hidden state)
    c1 = all_c_1.pop(-1)
    print(f"Stage 2 -- pop(-1):       c = {c1.shape}  (live state c_0)")
    print(f"           remaining:     len={len(all_c_1)}  (accumulated skips)")
    assert c1.shape == torch.Size([B, S, D])
    assert len(all_c_1) == 1  # only c_skip_0 remains

    # Stage 3: super().forward()
    c1 = DiTBlock.forward(block1, c1, context, t_mod, freqs)
    print(f"Stage 3 -- super().forward(): {c1.shape}")
    assert c1.shape == torch.Size([B, S, D])

    # Stage 4: after_proj
    c_skip_1 = block1.after_proj(c1)
    print(f"Stage 4 -- after_proj(c):     {c_skip_1.shape}  (c_skip_1)")
    assert c_skip_1.shape == torch.Size([B, S, D])

    # Stage 5: append
    all_c_1 += [c_skip_1, c1]
    print(f"Stage 5 -- all_c += [...]:    len={len(all_c_1)}  [c_skip_0, c_skip_1, c_1]")
    assert len(all_c_1) == 3

    # Stage 6: stack
    c_stacked_1 = torch.stack(all_c_1)
    print(f"Stage 6 -- stack(all_c):      {c_stacked_1.shape}")
    assert c_stacked_1.shape == torch.Size([3, B, S, D])

print(f"\nBlock 1 complete: stack grew from [2,...] to {c_stacked_1.shape}")

In [ ]:
# Source: diffsynth/models/wan_video_vace.py, lines 13-24
# ---- Block 2 (block_id=2): same unbind + pop pattern, stack grows to [4,...] ----
block2 = VaceWanAttentionBlock(has_image_input=False, dim=dim, num_heads=num_heads, ffn_dim=ffn_dim, block_id=2)
block2.eval()
with torch.no_grad():
    # Stage 1: unbind the stack from Block 1
    all_c_2 = list(torch.unbind(c_stacked_1))
    print(f"Stage 1 -- unbind(c):     {len(all_c_2)} tensors, each {all_c_2[0].shape}")
    assert len(all_c_2) == 3  # [c_skip_0, c_skip_1, c_1]

    # Stage 2: pop last element (live hidden state)
    c2 = all_c_2.pop(-1)
    print(f"Stage 2 -- pop(-1):       c = {c2.shape}  (live state c_1)")
    print(f"           remaining:     len={len(all_c_2)}  (accumulated skips)")
    assert c2.shape == torch.Size([B, S, D])
    assert len(all_c_2) == 2  # c_skip_0, c_skip_1 remain

    # Stage 3: super().forward()
    c2 = DiTBlock.forward(block2, c2, context, t_mod, freqs)
    print(f"Stage 3 -- super().forward(): {c2.shape}")
    assert c2.shape == torch.Size([B, S, D])

    # Stage 4: after_proj
    c_skip_2 = block2.after_proj(c2)
    print(f"Stage 4 -- after_proj(c):     {c_skip_2.shape}  (c_skip_2)")
    assert c_skip_2.shape == torch.Size([B, S, D])

    # Stage 5: append
    all_c_2 += [c_skip_2, c2]
    print(f"Stage 5 -- all_c += [...]:    len={len(all_c_2)}  [c_skip_0, c_skip_1, c_skip_2, c_2]")
    assert len(all_c_2) == 4

    # Stage 6: stack
    c_stacked_2 = torch.stack(all_c_2)
    print(f"Stage 6 -- stack(all_c):      {c_stacked_2.shape}")
    assert c_stacked_2.shape == torch.Size([4, B, S, D])

print(f"\nBlock 2 complete: stack grew from [3,...] to {c_stacked_2.shape}")

In [ ]:
# Source: diffsynth/models/wan_video_vace.py, line 86-87
# FINAL: extract hints by dropping the last element (live hidden state)
hints = list(torch.unbind(c_stacked_2))[:-1]
print(f"Final stack: {c_stacked_2.shape}  = [c_skip_0, c_skip_1, c_skip_2, c_2]")
print(f"hints = unbind(c)[:-1]  -> {len(hints)} hints (dropped final live state c_2)")
for i, h in enumerate(hints):
    assert h.shape == torch.Size([B, S, D])
    print(f"  hint[{i}]: {h.shape}")

# Key verification: number of hints == number of VACE blocks
assert len(hints) == 3, f"Expected 3 hints (one per VACE block), got {len(hints)}"
print(f"\nlen(hints) == num_vace_blocks == 3  (one hint per block)")

## Full VaceWanModel Forward -- End-to-End Verification

Now we instantiate the complete `VaceWanModel` with reduced config and run a full forward pass to verify the accumulation produces the expected number of hints.

In [ ]:
# Source: diffsynth/models/wan_video_vace.py, lines 27-51
vace_in_dim = 16  # reduced from production 96 (D-06)
vace_layers = (0, 2, 4)  # 3 VACE blocks (D-05)
patch_size = (1, 2, 2)

vace = VaceWanModel(
    vace_layers=vace_layers, vace_in_dim=vace_in_dim, patch_size=patch_size,
    has_image_input=False, dim=dim, num_heads=num_heads, ffn_dim=ffn_dim,
)
vace.eval()
total_params = sum(p.numel() for p in vace.parameters())
print(f"VaceWanModel: {total_params:,} parameters")
print(f"  vace_layers: {vace_layers}")
print(f"  vace_in_dim: {vace_in_dim}")
print(f"  vace_layers_mapping: {vace.vace_layers_mapping}")

# Source: diffsynth/models/wan_video_vace.py, line 42
# vace_layers_mapping: maps FROM DiT block_id TO VACE hint index
mapping = {i: n for n, i in enumerate(vace_layers)}
assert mapping == {0: 0, 2: 1, 4: 2}
assert vace.vace_layers_mapping == mapping
print(f"\nvace_layers_mapping: {mapping}")
print(f"  DiT layer 0 -> hint[0], DiT layer 2 -> hint[1], DiT layer 4 -> hint[2]")

# Full forward pass
F_vid, H_vid, W_vid = 4, 8, 8
vace_context = [torch.randn(vace_in_dim, F_vid, H_vid, W_vid)]
x_full = torch.randn(B, S, D)

with torch.no_grad():
    hints_full = vace(x_full, vace_context, context, t_mod, freqs)

assert len(hints_full) == len(vace_layers)
for i, h in enumerate(hints_full):
    assert h.shape == torch.Size([B, S, D])
    print(f"hint[{i}]: {h.shape}")
print(f"\nVaceWanModel produces {len(hints_full)} hints, matching {len(vace_layers)} vace_layers")

## Summary

### Key Takeaways
- **VaceWanAttentionBlock** adds only 2 Linear layers to DiTBlock: `before_proj` (block_id=0 only, projects control tokens + adds noisy latent) and `after_proj` (all blocks, projects skip connections)
- **The accumulation pattern** grows the stack by 1 element per block: `[2,B,S,D]` -> `[3,B,S,D]` -> `[4,B,S,D]`. The last element is always the live hidden state; everything before it is accumulated skip projections
- **Final hints** are extracted by `unbind(stack)[:-1]`, dropping the live hidden state. This yields exactly one hint per VACE block (3 blocks -> 3 hints)

### Source References

| Symbol | Location |
|--------|----------|
| VaceWanAttentionBlock.__init__ | `diffsynth/models/wan_video_vace.py`, line 5 |
| VaceWanAttentionBlock.forward | `diffsynth/models/wan_video_vace.py`, line 13 |
| VaceWanModel.__init__ | `diffsynth/models/wan_video_vace.py`, line 27 |
| VaceWanModel.forward | `diffsynth/models/wan_video_vace.py`, line 53 |
| DiTBlock (base class) | `diffsynth/models/wan_video_dit.py`, line 197 |

### Next
NB-12 shows how these hints are injected into the DiT during inference: pre-computed once by VaceWanModel, then added at matching even-indexed DiT layers as `x = x + hint * vace_scale`.

## Exercises

### Exercise 1 -- What if VACE used 5 blocks instead of 3?
Predict the final stack shape if `vace_layers = (0, 2, 4, 6, 8)` (5 VACE blocks). Then modify the VaceWanModel instantiation cell above to use 5 blocks and verify your prediction. How many hints does `unbind(c)[:-1]` produce? Does the formula `len(hints) == len(vace_layers)` still hold?

### Exercise 2 -- Remove after_proj from block_id=1
Create a modified VaceWanAttentionBlock where block_id=1's `after_proj` is replaced with `nn.Identity()`. Run the 3-block forward trace again. How does this change what gets accumulated in the stack vs what flows forward as the live hidden state? Is `c_skip_1` now identical to `c_1`?

### Exercise 3 -- VACE parameter overhead
Compare parameter counts: how many parameters does VACE add beyond the base DiT blocks it reuses? For 3 VACE blocks with `dim=384`: count `before_proj` (1 block only) + `after_proj` (3 blocks) + `vace_patch_embedding` (Conv3d). Express this as a percentage of the total VaceWanModel parameter count. How lightweight is the VACE control overhead?